# NB02 — Tokenisation, PyTorch Dataset et Split 90/5/5
> **Phase B** : Préparation des données pour l'apprentissage statistique.


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres généraux (Settings) et reproductibilité
# ==============================================================================
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from utils import set_seed, AminoAcidTokenizer, SequenceDataset, collate_fn

SEED = 42
set_seed(SEED)

BATCH_SIZE = 64
MIN_LEN = 100
MAX_LEN = 140
SPLIT_RATIOS = (0.90, 0.05, 0.05)

DATA_PATH = ROOT_DIR / "data" / "vhh_clean.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT_DIR / "data" / "toy" / "vhh_clean.csv"

print(f"Seed: {SEED} | Batch size: {BATCH_SIZE} | Source: {DATA_PATH}")


In [ ]:
# ==============================================================================
# Cellule 2 : Tokeniseur par caractère (Alphabet des 20 acides aminés)
# ==============================================================================
tokenizer = AminoAcidTokenizer()
print(f"Taille du vocabulaire : {tokenizer.vocab_size} tokens")
print(f"Tokens : {tokenizer.vocab}")

test_seq = "QVQLVESGGGLVQPGGSLRLSCAASGFTFS"
encoded = tokenizer.encode(test_seq, add_special_tokens=True)
decoded = tokenizer.decode(encoded)
print(f"Test encodage : {encoded}")
print(f"Test décodage : {decoded}")
assert decoded == test_seq, "Erreur dans le tokeniseur !"


In [ ]:
# ==============================================================================
# Cellule 3 : SequenceDataset et DataLoader avec Collate
# ==============================================================================
from torch.utils.data import DataLoader

df = pd.read_csv(DATA_PATH)
sequences = df["sequence"].tolist()
print(f"Séquences chargées : {len(sequences):,}")

dataset = SequenceDataset(sequences, tokenizer)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_id)
)

for inputs, targets in loader:
    print(f"Batch inputs  : {inputs.shape} [dtype: {inputs.dtype}]")
    print(f"Batch targets : {targets.shape} [dtype: {targets.dtype}]")
    break


In [ ]:
# ==============================================================================
# Cellule 4 : Split aléatoire 90% / 5% / 5% (Plan du professeur)
# ==============================================================================
n = len(sequences)
indices = np.random.permutation(n)
n_train = int(SPLIT_RATIOS[0] * n)
n_val = int(SPLIT_RATIOS[1] * n)

train_seqs = [sequences[i] for i in indices[:n_train]]
val_seqs = [sequences[i] for i in indices[n_train:n_train + n_val]]
test_seqs = [sequences[i] for i in indices[n_train + n_val:]]

print(f"Train : {len(train_seqs):,} ({len(train_seqs)/n:.1%})")
print(f"Val   : {len(val_seqs):,} ({len(val_seqs)/n:.1%})")
print(f"Test  : {len(test_seqs):,} ({len(test_seqs)/n:.1%})")

# Vérification stricte de disjonction
assert set(train_seqs).isdisjoint(set(val_seqs)), "Fuite entre train et val !"
assert set(train_seqs).isdisjoint(set(test_seqs)), "Fuite entre train et test !"
assert set(val_seqs).isdisjoint(set(test_seqs)), "Fuite entre val et test !"
print("Vérification : Les trois ensembles sont strictement disjoints.")

out_dir = ROOT_DIR / "data"
out_dir.mkdir(exist_ok=True)
pd.DataFrame({"sequence": train_seqs}).to_csv(out_dir / "train.csv", index=False)
pd.DataFrame({"sequence": val_seqs}).to_csv(out_dir / "val.csv", index=False)
pd.DataFrame({"sequence": test_seqs}).to_csv(out_dir / "test.csv", index=False)
print("Splits sauvegardés dans data/.")
